# Molecular Graph Learning Curve

This notebook measures how predictive performance on the MoleculeNet HIV dataset changes as the number of streamed training graphs increases. It uses `NSPPKClassifier.partial_fit_from(...)` with NSPPK features configured as `radius=1`, `distance=4`, and `connector=1`, plus a selectable incremental classifier (`SGDClassifier`, `MLPClassifier`, or a custom estimator that supports `partial_fit(...)`).

A fixed balanced test sample is loaded once. Each repeat streams a random Bernoulli sample from source rows after `TEST_PREFIX_SIZE`, while scoring at exponentially growing train sizes. The wrapper fits any learned attribute preprocessing on the first training batch, then holds it fixed while it incrementally trains and reports checkpoints through `batch_callback`.


In [1]:
from pathlib import Path
import csv
import gc
import os
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rdkit import RDLogger
from sklearn.base import clone
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.neural_network import MLPClassifier

try:
    import psutil
except ImportError:
    psutil = None

REPO_CANDIDATES = [Path.cwd().resolve(), Path.cwd().resolve().parent]
REPO_ROOT = next(candidate for candidate in REPO_CANDIDATES if (candidate / 'src').exists())
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from nsppk import NSPPK, NSPPKClassifier
from utils import plot_series_with_band_loess

DATASET_FILE = REPO_ROOT / 'data' / 'HIV.csv'
NBIT = 14
RADIUS = 1
DISTANCE = 4
CONNECTOR = 1
LIMIT = 0.99
TEST_PREFIX_SIZE = 2000
BALANCE_TEST_SET = True
TRAIN_SIZE_VALUES = None
N_REPEATS = 3
BATCH_SIZE = 512
PARALLEL = True
RANDOM_STATE = 42
CLASSIFIER_CHOICE = 'sgd'  # 'sgd', 'mlp', or 'generic'
CLASSIFIER_PARAMS = {} #{'hidden_layer_sizes': (256,64,)}
GENERIC_PARTIAL_FIT_CLASSIFIER = None
CLASSIFIER_REQUIRES_DENSE = CLASSIFIER_CHOICE == 'mlp'

RDLogger.DisableLog('rdApp.*')

def current_rss_mb():
    if psutil is not None:
        return psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2)
    import resource
    rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    if sys.platform == 'darwin':
        return rss / (1024 ** 2)
    return rss / 1024

MEMORY_LOG = []
MEMORY_PEAK_MB = 0.0

def record_memory(stage, emit=True):
    global MEMORY_PEAK_MB
    rss_mb = float(current_rss_mb())
    MEMORY_PEAK_MB = max(MEMORY_PEAK_MB, rss_mb)
    event = {'stage': stage, 'rss_mb': rss_mb, 'peak_rss_mb': MEMORY_PEAK_MB}
    MEMORY_LOG.append(event)
    if emit:
        print(f"[memory] {stage:<28} rss={rss_mb:8.2f} MB  peak={MEMORY_PEAK_MB:8.2f} MB")
    return event

def count_dataset_rows(dataset_file):
    with Path(dataset_file).open() as stream:
        header_seen = False
        count = 0
        for line in stream:
            if not line.strip():
                continue
            if not header_seen:
                header_seen = True
                continue
            count += 1
        return count

def resolve_limit_count(limit, available_count):
    if limit is None:
        return int(available_count)
    if isinstance(limit, int):
        return min(int(limit), int(available_count))
    return min(int(np.floor(float(limit) * available_count)), int(available_count))

def make_halving_train_sizes(dataset_size, train_limit, min_size=BATCH_SIZE):
    if train_limit <= 0:
        return []
    if TRAIN_SIZE_VALUES is not None:
        train_sizes = [n for n in TRAIN_SIZE_VALUES if n <= train_limit]
        if train_limit not in train_sizes:
            train_sizes.append(train_limit)
        return sorted(set(train_sizes))
    train_sizes = []
    current = dataset_size // 2
    while current >= min_size:
        if current <= train_limit:
            train_sizes.append(current)
        current //= 2
    train_sizes.append(train_limit)
    return sorted(set(train_sizes))

def compute_balanced_class_weight(labels, classes):
    labels = np.asarray(labels)
    classes = np.asarray(classes)
    n_samples = labels.shape[0]
    n_classes = classes.shape[0]
    weights = {}
    for cls in classes:
        class_count = int(np.sum(labels == cls))
        if class_count == 0:
            raise ValueError(f'Cannot compute class weight: class {cls} is absent from the reference labels.')
        weights[int(cls)] = n_samples / (n_classes * class_count)
    return weights

def load_labels_from_csv(dataset_file, label_column, start_after_instance=0):
    labels = []
    with Path(dataset_file).open(newline='') as stream:
        reader = csv.DictReader(line for line in stream if line.strip())
        for row_idx, row in enumerate(reader):
            if row_idx < start_after_instance:
                continue
            labels.append(int(row[label_column]))
    return np.asarray(labels)

def build_incremental_estimator(choice, random_state, class_weight=None, classifier_params=None, generic_classifier=None):
    classifier_params = dict(classifier_params or {})
    if choice == 'sgd':
        default_params = {
            'loss': 'log_loss',
            'alpha': 1e-5,
            'penalty': 'l2',
            'random_state': random_state,
            'class_weight': class_weight,
        }
        default_params.update(classifier_params)
        classifier = SGDClassifier(**default_params)
    elif choice == 'mlp':
        if class_weight is not None:
            raise ValueError('MLPClassifier does not accept class_weight; set CLASSIFIER_CHOICE to \'sgd\' or use a generic classifier instead.')
        default_params = {
            'hidden_layer_sizes': (256,),
            'alpha': 1e-4,
            'learning_rate_init': 1e-3,
            'max_iter': 1,
            'random_state': random_state,
        }
        default_params.update(classifier_params)
        classifier = MLPClassifier(**default_params)
    elif choice == 'generic':
        if generic_classifier is None:
            raise ValueError('Set GENERIC_PARTIAL_FIT_CLASSIFIER to an estimator instance or zero-argument factory when CLASSIFIER_CHOICE="generic".')
        classifier = generic_classifier() if callable(generic_classifier) else clone(generic_classifier)
    else:
        raise ValueError(f'Unknown CLASSIFIER_CHOICE: {choice!r}')

    if not hasattr(classifier, 'partial_fit'):
        raise TypeError(f'{type(classifier).__name__} does not implement partial_fit(...)')
    return classifier

def build_nsppk_classifier(choice, random_state, class_weight=None, classifier_params=None, generic_classifier=None):
    estimator = build_incremental_estimator(
        choice,
        random_state=random_state,
        class_weight=class_weight if choice != 'mlp' else None,
        classifier_params=classifier_params,
        generic_classifier=generic_classifier,
    )
    return NSPPKClassifier(
        estimator=estimator,
        radius=RADIUS,
        distance=DISTANCE,
        connector=CONNECTOR,
        nbits=NBIT,
        dense=CLASSIFIER_REQUIRES_DENSE,
        parallel=PARALLEL,
    )

def get_positive_scores(classifier, X):
    if hasattr(classifier, 'decision_function'):
        scores = classifier.decision_function(X)
    elif hasattr(classifier, 'predict_proba'):
        scores = classifier.predict_proba(X)
    else:
        raise TypeError(f'{type(classifier).__name__} does not expose decision_function(...) or predict_proba(...) for scoring')

    scores = np.asarray(scores)
    if scores.ndim == 2:
        if scores.shape[1] < 2:
            raise ValueError(f'Expected binary-class scores with at least 2 columns, got shape {scores.shape}')
        scores = scores[:, 1]
    return scores.ravel()


## Setup

Load the fixed balanced test sample and define the post-prefix source range used for training. The supervised wrapper fits its own encoder preprocessing on each repeat’s first training batch. The checkpoint schedule is derived by repeatedly halving the dataset size down to one `BATCH_SIZE`, retaining values within the sampled training limit.


In [2]:
loader = NSPPK(radius=RADIUS, distance=DISTANCE, connector=CONNECTOR, nbits=NBIT, parallel=PARALLEL)
record_memory('start')

test_graphs = loader.load_from(
    DATASET_FILE,
    'smiles',
    limit=TEST_PREFIX_SIZE,
    balance=BALANCE_TEST_SET,
    random_state=RANDOM_STATE,
    label_extractor=lambda graph: int(graph.graph['HIV_active']),
)
y_test = np.asarray([int(graph.graph['HIV_active']) for graph in test_graphs])
record_memory('test set loaded')

dataset_size = count_dataset_rows(DATASET_FILE)
available_train_pool = max(dataset_size - TEST_PREFIX_SIZE, 0)
train_limit = resolve_limit_count(LIMIT, available_train_pool)

classes = np.array([0, 1])
train_pool_labels = load_labels_from_csv(DATASET_FILE, 'HIV_active', start_after_instance=TEST_PREFIX_SIZE)
class_weight = compute_balanced_class_weight(train_pool_labels, classes)
classifier_label = CLASSIFIER_CHOICE if CLASSIFIER_CHOICE != 'generic' else type(GENERIC_PARTIAL_FIT_CLASSIFIER).__name__ if GENERIC_PARTIAL_FIT_CLASSIFIER is not None and not callable(GENERIC_PARTIAL_FIT_CLASSIFIER) else 'generic'

train_size_values = make_halving_train_sizes(dataset_size, train_limit)

print('test graphs:', len(test_graphs))
print(f'test positive rate: {y_test.mean():.3f}')
print('dataset size:', dataset_size)
print('available train pool:', available_train_pool)
print('train pool limit:', train_limit)
print('classifier choice:', classifier_label)
print('class weight for partial_fit:', class_weight)
print('classifier requires dense input:', CLASSIFIER_REQUIRES_DENSE)
print('stream batch size:', BATCH_SIZE)
print('train sizes:', train_size_values)
print(f'initial peak RSS: {MEMORY_PEAK_MB:.2f} MB')


[memory] start                        rss=  305.55 MB  peak=  305.55 MB
[memory] test set loaded              rss= 1220.11 MB  peak= 1220.11 MB
test graphs: 2000
test positive rate: 0.500
dataset size: 41127
available train pool: 39127
train pool limit: 38735
classifier choice: sgd
class weight for partial_fit: {0: 0.5180188529365037, 1: 14.374357090374724}
classifier requires dense input: False
stream batch size: 512
train sizes: [642, 1285, 2570, 5140, 10281, 20563, 38735]
initial peak RSS: 1220.11 MB


## Incremental Learning Curve

Each repeat trains an `NSPPKClassifier` through `partial_fit_from(...)`. Its `batch_callback` scores the wrapper on the fixed held-out graphs whenever a cumulative training-size checkpoint is crossed. The first batch initializes preprocessing and is included in predictor training; all later batches use the same encoder state.


In [ ]:
results = []

print(f"{'train':>8} | {'repeat':>6} | {'roc_auc':>8} | {'avg_prec':>8} | {'seconds':>8} | {'rss_mb':>8} | {'peak_mb':>8}")
print('-' * 79)

repeat_histories = {
    train_size: {'roc_auc': [], 'avg_precision': [], 'runtime_sec': [], 'rss_mb': [], 'peak_rss_mb': []}
    for train_size in train_size_values
}
final_repeat_history = {
    'train_size': [], 'roc_auc': [], 'avg_precision': [], 'runtime_sec': [], 'rss_mb': [], 'peak_rss_mb': [],
}

def ensure_history(train_size):
    if train_size not in repeat_histories:
        repeat_histories[train_size] = {
            'roc_auc': [], 'avg_precision': [], 'runtime_sec': [], 'rss_mb': [], 'peak_rss_mb': [],
        }

def record_checkpoint(train_size, classifier, X_test_checkpoint, repeat_idx, t0, history=None):
    if history is None:
        ensure_history(train_size)
        history = repeat_histories[train_size]
    memory_event = record_memory(f'repeat {repeat_idx} train {train_size}', emit=False)
    y_score = get_positive_scores(classifier, X_test_checkpoint)
    runtime_sec = time.perf_counter() - t0
    roc_auc = roc_auc_score(y_test, y_score)
    avg_precision = average_precision_score(y_test, y_score)

    if 'train_size' in history:
        history['train_size'].append(int(train_size))
    history['roc_auc'].append(float(roc_auc))
    history['avg_precision'].append(float(avg_precision))
    history['runtime_sec'].append(float(runtime_sec))
    history['rss_mb'].append(float(memory_event['rss_mb']))
    history['peak_rss_mb'].append(float(memory_event['peak_rss_mb']))
    print(
        f"{train_size:>8d} | {repeat_idx:>6d} | {roc_auc:>8.4f} | {avg_precision:>8.4f} | "
        f"{runtime_sec:>8.2f} | {memory_event['rss_mb']:>8.2f} | {memory_event['peak_rss_mb']:>8.2f}"
    )

for repeat_idx in range(1, N_REPEATS + 1):
    classifier = build_nsppk_classifier(
        CLASSIFIER_CHOICE,
        random_state=RANDOM_STATE + repeat_idx,
        class_weight=class_weight,
        classifier_params=CLASSIFIER_PARAMS,
        generic_classifier=GENERIC_PARTIAL_FIT_CLASSIFIER,
    )
    t0 = time.perf_counter()
    checkpoint_idx = [0]
    last_recorded_train_size = [None]
    trained_count = [0]
    test_features = [None]

    def checkpoint_callback(model, X_batch, y_batch, n_samples_seen):
        trained_count[0] = n_samples_seen
        if test_features[0] is None:
            test_features[0] = model.encoder_.transform(test_graphs)
        while (checkpoint_idx[0] < len(train_size_values)
               and n_samples_seen >= train_size_values[checkpoint_idx[0]]):
            stop = train_size_values[checkpoint_idx[0]]
            record_checkpoint(stop, model.estimator_, test_features[0], repeat_idx, t0)
            last_recorded_train_size[0] = stop
            checkpoint_idx[0] += 1

    classifier.partial_fit_from(
        DATASET_FILE,
        'smiles',
        limit=LIMIT,
        random_state=RANDOM_STATE + repeat_idx,
        batch_size=BATCH_SIZE,
        label_extractor=lambda graph: int(graph.graph['HIV_active']),
        start_after_instance=TEST_PREFIX_SIZE,
        classes=classes,
        batch_callback=checkpoint_callback,
        verbose=False,
    )

    if trained_count[0] > 0 and trained_count[0] != last_recorded_train_size[0]:
        record_checkpoint(trained_count[0], classifier.estimator_, test_features[0], repeat_idx, t0, history=final_repeat_history)
    print(f'repeat {repeat_idx} sampled train graphs: {trained_count[0]}')

reached_train_sizes = []
skipped_train_sizes = []
for train_size in sorted(repeat_histories):
    history = repeat_histories[train_size]
    if not history['roc_auc']:
        if train_size == train_limit and final_repeat_history['roc_auc']:
            continue
        skipped_train_sizes.append(train_size)
        continue
    reached_train_sizes.append(train_size)
    results.append({
        'train_size': train_size,
        'n_repeats_reached': len(history['roc_auc']),
        'mean_roc_auc': float(np.mean(history['roc_auc'])),
        'std_roc_auc': float(np.std(history['roc_auc'], ddof=0)),
        'mean_avg_precision': float(np.mean(history['avg_precision'])),
        'std_avg_precision': float(np.std(history['avg_precision'], ddof=0)),
        'mean_runtime_sec': float(np.mean(history['runtime_sec'])),
        'std_runtime_sec': float(np.std(history['runtime_sec'], ddof=0)),
        'mean_rss_mb': float(np.mean(history['rss_mb'])),
        'mean_peak_rss_mb': float(np.mean(history['peak_rss_mb'])),
        'classifier': classifier_label,
        'nbits': NBIT,
        'radius': RADIUS,
        'distance': DISTANCE,
        'connector': CONNECTOR,
    })

if final_repeat_history['roc_auc']:
    final_train_size = int(round(np.mean(final_repeat_history['train_size'])))
    reached_train_sizes.append(final_train_size)
    results.append({
        'train_size': final_train_size,
        'n_repeats_reached': len(final_repeat_history['roc_auc']),
        'mean_roc_auc': float(np.mean(final_repeat_history['roc_auc'])),
        'std_roc_auc': float(np.std(final_repeat_history['roc_auc'], ddof=0)),
        'mean_avg_precision': float(np.mean(final_repeat_history['avg_precision'])),
        'std_avg_precision': float(np.std(final_repeat_history['avg_precision'], ddof=0)),
        'mean_runtime_sec': float(np.mean(final_repeat_history['runtime_sec'])),
        'std_runtime_sec': float(np.std(final_repeat_history['runtime_sec'], ddof=0)),
        'mean_rss_mb': float(np.mean(final_repeat_history['rss_mb'])),
        'mean_peak_rss_mb': float(np.mean(final_repeat_history['peak_rss_mb'])),
        'classifier': classifier_label,
        'nbits': NBIT,
        'radius': RADIUS,
        'distance': DISTANCE,
        'connector': CONNECTOR,
    })

results_df = pd.DataFrame(results)
if skipped_train_sizes:
    print('skipped train sizes with no reached repeats:', skipped_train_sizes)
print('summarized train sizes:', reached_train_sizes)
print(f'final peak RSS: {MEMORY_PEAK_MB:.2f} MB')
results_df


   train | repeat |  roc_auc | avg_prec |  seconds |   rss_mb |  peak_mb
-------------------------------------------------------------------------------
     642 |      1 |   0.5599 |   0.6215 |     3.68 |  1164.97 |  1220.11
    1285 |      1 |   0.5180 |   0.5828 |     4.18 |  1198.66 |  1220.11
    2570 |      1 |   0.5682 |   0.6312 |     6.09 |  1212.52 |  1220.11
    5140 |      1 |   0.5759 |   0.6162 |     9.05 |  1235.31 |  1235.31
   10281 |      1 |   0.6641 |   0.6732 |    15.54 |  1238.36 |  1238.36
   20563 |      1 |   0.7912 |   0.7987 |    29.12 |  1242.06 |  1242.06
   38715 |      1 |   0.8176 |   0.8285 |    53.17 |  1254.08 |  1254.08
repeat 1 sampled train graphs: 38715
     642 |      2 |   0.5659 |   0.5860 |     3.65 |  1316.98 |  1316.98
    1285 |      2 |   0.5473 |   0.5910 |     4.30 |  1317.86 |  1317.86
    2570 |      2 |   0.5516 |   0.6200 |     6.31 |  1319.48 |  1319.48
    5140 |      2 |   0.5205 |   0.5560 |     9.40 |  1321.20 |  1321.20
   1028

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
frac = 0.8  # Local quadratic LOESS span; adjust between 0 and 1 for more or less smoothing.
x = results_df['train_size'].to_numpy()

plot_series_with_band_loess(
    axes[0],
    x,
    results_df['mean_roc_auc'],
    y_std=results_df['std_roc_auc'],
    frac=frac,
    label='Mean ROC-AUC',
)
axes[0].set_xlabel('Training graphs')
axes[0].set_ylabel('ROC-AUC')
axes[0].set_title('Learning curve: ROC-AUC')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

plot_series_with_band_loess(
    axes[1],
    x,
    results_df['mean_avg_precision'],
    y_std=results_df['std_avg_precision'],
    frac=frac,
    color='tab:green',
    label='Mean average precision',
)
axes[1].set_xlabel('Training graphs')
axes[1].set_ylabel('Average precision')
axes[1].set_title('Learning curve: average precision')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plot_series_with_band_loess(
    axes[2],
    x,
    results_df['mean_runtime_sec'],
    y_std=results_df['std_runtime_sec'],
    frac=frac,
    color='tab:red',
    label='Mean runtime',
)
axes[2].set_xlabel('Training graphs')
axes[2].set_ylabel('Seconds')
axes[2].set_title('End-to-end runtime')
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()


## Minimal Full-Stream `partial_fit_from` Example

For label-only graph inputs, call `NSPPKClassifier.partial_fit_from(...)` directly. It uses the built-in source reader, batches the graphs, and performs incremental estimator updates internally. This example trains on the entire file and is separate from the held-out learning-curve evaluation above.


In [ ]:
%%time

stream_model = build_nsppk_classifier(
    CLASSIFIER_CHOICE,
    random_state=RANDOM_STATE,
    class_weight=class_weight,
    classifier_params=CLASSIFIER_PARAMS,
    generic_classifier=GENERIC_PARTIAL_FIT_CLASSIFIER,
)
stream_model.partial_fit_from(
    DATASET_FILE,
    'smiles',
    batch_size=BATCH_SIZE,
    label_extractor=lambda graph: int(graph.graph['HIV_active']),
    classes=classes,
)

stream_model


## Partial-Fit vs Full-Fit Performance

Compare the wrapper’s `partial_fit_from(...)` path with the estimator’s ordinary `fit(...)` on the exact same sampled training features and fixed held-out test graphs. A batch callback collects the transformed feature batches for the full-fit estimator while recording the incremental model’s test scores. This comparison materializes the training feature matrix and therefore uses more memory than the streaming learning curve.

For `MLPClassifier`, ordinary `fit(...)` uses 200 epochs by default here (change `FIT_COMPARISON_MAX_ITER` below). The incremental model still updates once per streamed batch.


In [ ]:
from scipy.sparse import issparse, vstack

FIT_COMPARISON_MAX_ITER = 200
comparison_seed = RANDOM_STATE + 1
comparison_partial_model = build_nsppk_classifier(
    CLASSIFIER_CHOICE,
    random_state=comparison_seed,
    class_weight=class_weight,
    classifier_params=CLASSIFIER_PARAMS,
    generic_classifier=GENERIC_PARTIAL_FIT_CLASSIFIER,
)
comparison_X_batches = []
comparison_y_batches = []
comparison_train_started = time.perf_counter()

def collect_comparison_features(model, X_batch, y_batch, n_samples_seen):
    comparison_X_batches.append(X_batch)
    comparison_y_batches.append(y_batch.copy())

comparison_partial_model.partial_fit_from(
    DATASET_FILE,
    'smiles',
    limit=LIMIT,
    random_state=comparison_seed,
    batch_size=BATCH_SIZE,
    label_extractor=lambda graph: int(graph.graph['HIV_active']),
    start_after_instance=TEST_PREFIX_SIZE,
    classes=classes,
    batch_callback=collect_comparison_features,
)
comparison_partial_total_sec = time.perf_counter() - comparison_train_started

if not comparison_X_batches:
    raise RuntimeError('The sampled training stream was empty; cannot compare fit methods.')
if issparse(comparison_X_batches[0]):
    X_fit_comparison = vstack(comparison_X_batches, format='csr')
else:
    X_fit_comparison = np.concatenate(comparison_X_batches, axis=0)
y_fit_comparison = np.concatenate(comparison_y_batches)
comparison_train_count = len(y_fit_comparison)
del comparison_X_batches, comparison_y_batches
_ = gc.collect()
record_memory('fit comparison matrix ready')

comparison_fit_model = clone(comparison_partial_model.estimator_)
if isinstance(comparison_fit_model, MLPClassifier) and 'max_iter' not in CLASSIFIER_PARAMS:
    comparison_fit_model.set_params(max_iter=FIT_COMPARISON_MAX_ITER)
fit_started = time.perf_counter()
comparison_fit_model.fit(X_fit_comparison, y_fit_comparison)
comparison_fit_runtime_sec = time.perf_counter() - fit_started
record_memory('fit comparison trained')

X_test_comparison = comparison_partial_model.encoder_.transform(test_graphs)
partial_scores = get_positive_scores(comparison_partial_model.estimator_, X_test_comparison)
fit_scores = get_positive_scores(comparison_fit_model, X_test_comparison)
comparison_metrics = pd.DataFrame([
    {
        'training method': 'partial_fit (one streamed pass)',
        'training graphs': comparison_train_count,
        'ROC-AUC': roc_auc_score(y_test, partial_scores),
        'Average precision': average_precision_score(y_test, partial_scores),
        'training seconds': comparison_partial_total_sec,
    },
    {
        'training method': 'fit',
        'training graphs': comparison_train_count,
        'ROC-AUC': roc_auc_score(y_test, fit_scores),
        'Average precision': average_precision_score(y_test, fit_scores),
        'training seconds': comparison_fit_runtime_sec,
    },
])
display(comparison_metrics)

from sklearn.metrics import precision_recall_curve, roc_curve
fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
for label, scores, color in (
    ('partial_fit', partial_scores, 'tab:blue'),
    ('fit', fit_scores, 'tab:orange'),
):
    fpr, tpr, _ = roc_curve(y_test, scores)
    precision, recall, _ = precision_recall_curve(y_test, scores)
    roc_auc = roc_auc_score(y_test, scores)
    avg_precision = average_precision_score(y_test, scores)
    axes[0].plot(fpr, tpr, color=color, label=f'{label} (AUC={roc_auc:.3f})')
    axes[1].plot(recall, precision, color=color, label=f'{label} (AP={avg_precision:.3f})')
axes[0].plot([0, 1], [0, 1], linestyle=':', color='black', label='Random baseline')
axes[0].set(xlabel='False positive rate', ylabel='True positive rate', title='Held-out ROC')
axes[0].grid(alpha=0.25)
axes[0].legend()
axes[1].axhline(y_test.mean(), linestyle=':', color='black', label='Test prevalence')
axes[1].set(xlabel='Recall', ylabel='Precision', title='Held-out precision–recall', xlim=(0, 1), ylim=(0, 1.02))
axes[1].grid(alpha=0.25)
axes[1].legend()
comparison_metrics.set_index('training method')[['ROC-AUC', 'Average precision']].T.plot(
    kind='bar', ax=axes[2], color=['tab:blue', 'tab:orange'], ylim=(0, 1.02)
)
axes[2].set(title='Model comparison by metric', xlabel='', ylabel='Score')
axes[2].tick_params(axis='x', labelrotation=15)
axes[2].grid(axis='y', alpha=0.25)
axes[2].legend(loc='lower right')
fig.suptitle(f'{classifier_label}: fit vs partial_fit on the same {comparison_train_count:,} training molecules')
fig.tight_layout()
plt.show()

print('partial-fit wrapper:', comparison_partial_model)
print('fit estimator:', type(comparison_fit_model).__name__)
print('partial_fit time includes streamed feature generation; fit time is estimator training only.')
del X_fit_comparison, y_fit_comparison, X_test_comparison, comparison_partial_model, comparison_fit_model
_ = gc.collect()
